In [1]:
from datasets import load_dataset
import bpe
import itertools
from collections import Counter, defaultdict
from tqdm import tqdm 
import json
from tokenizers import Tokenizer

In [2]:
ds = load_dataset("stanfordnlp/sst2")

In [3]:

def pretokenize(s, begin_of_word=u"\u2581", replacement="_"):
    """This is close to the pre-tokenizer in the textbook"""
    return [("" if i == 0 else begin_of_word) + word.replace(begin_of_word, replacement) for i, word in enumerate(s.split())]

def prepare_corpus(corpus):
    return list(itertools.chain.from_iterable(pretokenize(s) for s in corpus))

def fit_bpe(prepared_corpus, max_merges, UNK=u"\u2047"):
    """
    Highly optimized BPE training using an inverted index and 
    linked-list position tracking. Integrated with tqdm for progress tracking.
    """
    print("Initializing vocabulary structures...")
    # 1. Count distinct word types to minimize duplicate work
    raw_word_counts = Counter(prepared_corpus)
    
    # 2. Build character vocabulary
    ch_vocab = set(ch for word in raw_word_counts for ch in word)
    vocabulary = set(ch_vocab)
    vocabulary.add(UNK)
    
    # 3. Represent words as doubly linked lists using arrays
    words_stats = []
    pair_to_word_pos = defaultdict(dict)
    pair_counts = Counter()
    
    for word, freq in raw_word_counts.items():
        w_len = len(word)
        if w_len == 0:
            continue
        
        symbols = list(word)
        prev_idx = list(range(-1, w_len - 1))
        next_idx = list(range(1, w_len + 1))
        next_idx[-1] = -1
        
        word_idx = len(words_stats)
        words_stats.append([symbols, prev_idx, next_idx, freq])
        
        # Populate initial pairs
        for i in range(w_len - 1):
            pair = (symbols[i], symbols[i+1])
            pair_to_word_pos[pair][(word_idx, i)] = freq
            pair_counts[pair] += freq

    merges = []

    print(f"Starting BPE learning for {max_merges} merges...")
    # 4. Main BPE Loop with tqdm progress bar
    with tqdm(total=max_merges, desc="Learning BPE Merges", unit="merge") as pbar:
        for n in range(max_merges):
            if not pair_counts:
                pbar.set_postfix_str("No more pairs to merge")
                break

            # Find the highest frequency pair. Tie-breaker: alphabetical order
            best_pair = min(
                pair_counts,
                key=lambda p: (-pair_counts[p], p)
            )
            
            # If the best pair has 0 or negative frequency, we are done
            if pair_counts[best_pair] <= 0:
                pbar.set_postfix_str("Frequency threshold reached")
                break

            merges.append(best_pair)
            merged_str = best_pair[0] + best_pair[1]
            vocabulary.add(merged_str)

            # Get all valid locations where this pair occurs
            locations = pair_to_word_pos[best_pair]
            del pair_counts[best_pair]
            del pair_to_word_pos[best_pair]

            # Filter out invalid/stale matches before processing
            valid_locations = []
            for (word_idx, pos), freq in locations.items():
                symbols, prev_idx, next_idx, _ = words_stats[word_idx]
                if next_idx[pos] != -1 and symbols[pos] == best_pair[0]:
                    r_idx = next_idx[pos]
                    if symbols[r_idx] == best_pair[1]:
                        valid_locations.append((word_idx, pos, freq))

            # Update the data structures for each valid occurrence
            for word_idx, l_idx, freq in valid_locations:
                symbols, prev_idx, next_idx, _ = words_stats[word_idx]
                
                # Verify it's still intact (handles overlapping pairs like 'aaa')
                r_idx = next_idx[l_idx]
                if r_idx == -1 or symbols[l_idx] != best_pair[0] or symbols[r_idx] != best_pair[1]:
                    continue
                    
                ll_idx = prev_idx[l_idx]
                rr_idx = next_idx[r_idx]

                # 1. Remove old pair to the left
                if ll_idx != -1:
                    old_left_pair = (symbols[ll_idx], symbols[l_idx])
                    pair_counts[old_left_pair] -= freq
                    pair_to_word_pos[old_left_pair].pop((word_idx, ll_idx), None)

                # 2. Remove old pair to the right
                if rr_idx != -1:
                    old_right_pair = (symbols[r_idx], symbols[rr_idx])
                    pair_counts[old_right_pair] -= freq
                    pair_to_word_pos[old_right_pair].pop((word_idx, r_idx), None)

                # 3. Merge the symbols in place
                symbols[l_idx] = merged_str

                # 4. Re-link the pointers bypassing r_idx
                next_idx[l_idx] = rr_idx
                if rr_idx != -1:
                    prev_idx[rr_idx] = l_idx

                # 5. Add new pair to the left
                if ll_idx != -1:
                    new_left_pair = (symbols[ll_idx], symbols[l_idx])
                    pair_counts[new_left_pair] += freq
                    pair_to_word_pos[new_left_pair][(word_idx, ll_idx)] = freq

                # 6. Add new pair to the right
                if rr_idx != -1:
                    new_right_pair = (symbols[l_idx], symbols[rr_idx])
                    pair_counts[new_right_pair] += freq
                    pair_to_word_pos[new_right_pair][(word_idx, l_idx)] = freq

            pbar.update(1)

    return {
        "merges": merges,
        "vocabulary": vocabulary,
        "character_vocab": ch_vocab,
        "UNK": UNK
    }

def segment_entry(entry, model):
    """Segment one pretokenized entry using a fitted BPE model."""
    ch_vocab = model["character_vocab"]
    
    symbols = [
        character if character in ch_vocab else model["UNK"]
        for character in entry
    ]

    for pair in model["merges"]:
        merged = []
        i = 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == pair[0] and symbols[i+1] == pair[1]:
                merged.append(pair[0] + pair[1])
                i += 2
            else:
                merged.append(symbols[i])
                i += 1
        symbols = merged

    return symbols

def segment_text(text, model):
    """Pretokenize and segment a raw text."""
    entries = pretokenize(text)
    return list(itertools.chain.from_iterable(segment_entry(entry, model) for entry in entries))


In [4]:
def save_to_huggingface_format(model, filepath, begin_of_word=u"\u2581"):
    """
    Slaat het BPE-model op zodat "hello world" via Hugging Face exact
    gesegmenteerd wordt in ['hello', ' world'] (met \u2581).
    Foutloos met de nieuwste prepend_scheme standaard van Hugging Face.
    """
    # 1. Hugging Face vereist een vocab dict van { "token": ID }
    sorted_vocab = sorted(list(model["vocabulary"]))
    hf_vocab = {token: idx for idx, token in enumerate(sorted_vocab)}
    
    # Zorg dat het losse prefix-karakter sowieso in de vocab staat
    if begin_of_word not in hf_vocab:
        hf_vocab[begin_of_word] = len(hf_vocab)
    
    # 2. Zet de merges om naar het juiste formaat ("token_a token_b")
    # We filteren witruimtes/lege waarden uit om de 'invalid merge line' te voorkomen
    hf_merges = []
    for pair in model["merges"]:
        left, right = pair[0], pair[1]
        if not left or not right or left == " " or right == " ":
            continue
        hf_merges.append(f"{left} {right}")
    
    # 3. Bouw de officiële Hugging Face JSON-architectuur op
    hf_json = {
        "version": "1.0",
        "truncation": None,
        "padding": None,
        "added_tokens": [],
        "normalizer": {
            "type": "Lowercase"
        },
        
        # FIX: We gebruiken 'prepend_scheme': 'first' in plaats van add_prefix_space.
        # Dit zorgt ervoor dat het allereerste woord GEEN prefix krijgt, 
        # maar dat spaties daarna netjes worden omgezet in de \u2581 prefix.
        "pre_tokenizer": {
            "type": "Sequence",
            "pretokenizers": [
                {
                    "type": "Punctuation"
                },
                {
                    "type": "Metaspace",
                    "replacement": begin_of_word,
                    "prepend_scheme": "never",
                    "split": True
                }
            ]
        },
        "post_processor": None,
        "decoder": {
            "type": "Metaspace",
            "replacement": begin_of_word,
            "prepend_scheme": "never",
            "split": True
        },
        "model": {
            "type": "BPE",
            "dropout": None,
            "unk_token": model["UNK"],
            "continuing_subword_prefix": None,
            "end_of_word_suffix": None,
            "fuse_unk": False,
            "vocab": hf_vocab,
            "merges": hf_merges
        }
    }
    
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(hf_json, f, ensure_ascii=False, indent=4)
    print(f"Model succesvol opgeslagen!")


In [5]:
text_segments = list(set([s.lower() for s in (
    list(ds["train"]["sentence"])
    + list(ds["validation"]["sentence"])
    + list(ds["test"]["sentence"])
)]))
print(f"Number of text segments: {len(text_segments)}")

Number of text segments: 69671


In [6]:
corpus = bpe.prepare_corpus(text_segments)

In [7]:
def add_pad(model, pad_token="[PAD]"):
    """
    Adds a padding token to the BPE model and updates the vocabulary.
    This is useful for models that require a padding token for batching.
    """
    if pad_token not in model["vocabulary"]:
        model["vocabulary"].add(pad_token)
        # Update the Hugging Face format if needed
        print(f"Padding token '{pad_token}' added to the model.")
    else:
        print(f"Padding token '{pad_token}' already exists in the model.")

In [8]:
for max_merges in [2000, 4000, 8000, 16000, 32000]:
    print(f"Fitting BPE with {max_merges} merges...")
    model = fit_bpe(corpus, max_merges=max_merges)
    add_pad(model)
    save_to_huggingface_format(model, f"results/bpe_{max_merges}.json")
    model = Tokenizer.from_file(f"results/bpe_{max_merges}.json")
    print(f"padding token: {model.token_to_id('[PAD]')}")
    

Fitting BPE with 2000 merges...
Initializing vocabulary structures...
Starting BPE learning for 2000 merges...


Learning BPE Merges: 100%|██████████| 2000/2000 [00:07<00:00, 270.03merge/s]


Padding token '[PAD]' added to the model.
Model succesvol opgeslagen!
padding token: 67
Fitting BPE with 4000 merges...
Initializing vocabulary structures...
Starting BPE learning for 4000 merges...


Learning BPE Merges: 100%|██████████| 4000/4000 [00:20<00:00, 195.94merge/s]


Padding token '[PAD]' added to the model.
Model succesvol opgeslagen!
padding token: 121
Fitting BPE with 8000 merges...
Initializing vocabulary structures...
Starting BPE learning for 8000 merges...


Learning BPE Merges: 100%|██████████| 8000/8000 [00:49<00:00, 161.78merge/s]


Padding token '[PAD]' added to the model.
Model succesvol opgeslagen!
padding token: 269
Fitting BPE with 16000 merges...
Initializing vocabulary structures...
Starting BPE learning for 16000 merges...


Learning BPE Merges: 100%|██████████| 16000/16000 [02:07<00:00, 125.81merge/s]


Padding token '[PAD]' added to the model.
Model succesvol opgeslagen!
padding token: 819
Fitting BPE with 32000 merges...
Initializing vocabulary structures...
Starting BPE learning for 32000 merges...


Learning BPE Merges: 100%|██████████| 32000/32000 [04:15<00:00, 125.47merge/s]


Padding token '[PAD]' added to the model.
Model succesvol opgeslagen!
padding token: 1847


In [9]:
for max_merges in [2000, 4000, 8000, 16000, 32000]:
    try:
        model = Tokenizer.from_file(f"results/bpe_{max_merges}.json")
        print(f"Successfully loaded tokenizer for {max_merges} merges.")
        print(model.token_to_id("[PAD]"))
        print(model.get_vocab_size())
        for i in range(16):
            print(ds['train']['sentence'][i])
            output = model.encode(ds['train']['sentence'][i])
            print(output.tokens)
            print(output.ids)
            print()
    except Exception as e:
        print(f"Error loading tokenizer for {max_merges} merges: {e}")

Successfully loaded tokenizer for 2000 merges.
67
2072
hide new secretions from the parental units 
['h', 'ide', '▁new', '▁sec', 'ret', 'ions', '▁from', '▁the', '▁parent', 'al', '▁un', 'its', '▁']
[383, 426, 1616, 1784, 756, 489, 1365, 1922, 1656, 103, 1981, 531, 962]

contains no wit , only labored gags 
['cont', 'ains', '▁no', '▁wit', '▁', ',', '▁only', '▁l', 'ab', 'ored', '▁gags', '▁']
[251, 97, 1620, 2042, 962, 12, 1639, 1510, 70, 662, 1373, 962]

that loves its characters and communicates something rather beautiful about human nature 
['that', '▁lo', 'ves', '▁its', '▁characters', '▁and', '▁comm', 'un', 'ic', 'ates', '▁something', '▁rather', '▁beautiful', '▁about', '▁human', '▁nature', '▁']
[812, 1537, 900, 1492, 1127, 1032, 1158, 856, 410, 173, 1841, 1734, 1069, 991, 1445, 1611, 962]

remains utterly satisfied to remain the same throughout 
['rem', 'ains', '▁utter', 'ly', '▁satisf', 'ied', '▁to', '▁rem', 'ain', '▁the', '▁same', '▁through', 'out', '▁']
[747, 97, 1991, 580, 1772, 42